# 02 – Zero-Shot VLM (Phase 3)
OpenCLIP ViT-L/14 (OpenAI weights) zero-shot with prompt ensembling + softmax-over-categories scoring.
Expected val R-precision ≥ 0.75.

**Needs GPU.** Features are cached so later notebooks run on CPU.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, "/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

import numpy as np
import pandas as pd
import torch

from src.config import seed_everything
seed_everything(42)

import torchvision, timm, sklearn
try:
    import open_clip
    print("open_clip:", open_clip.__version__)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "open-clip-torch", "-q"])
    import open_clip
    print("open_clip installed:", open_clip.__version__)

print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision,
    timm=timm, sklearn=sklearn, numpy=np).items()})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

DATA_DIR = pathlib.Path("/kaggle/input/competitions/match-the-dish-to-its-rizz")
OUT_DIR  = pathlib.Path("/kaggle/working")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print("DATA_DIR exists:", DATA_DIR.exists())

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
from src.data import load_csvs, parse_val_ground_truth

dfs          = load_csvs(DATA_DIR)
train_df     = dfs["train"]
val_gallery  = dfs["val_gallery"]
val_queries  = dfs["val_queries"]
val_gt_df    = dfs["val_gt"]
test_gallery = dfs["test_gallery"]
test_queries = dfs["test_queries"]
meta         = dfs["category_metadata"]

val_gt      = parse_val_ground_truth(val_gt_df)
cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {c: i for i, c in enumerate(cat_id_list)}
K           = len(cat_id_list)
print(f"K={K}")

In [ ]:
# ── Text matcher ─────────────────────────────────────────────────────────────
from src.text_matching import build_surface_form_dict, TextMatcher
surface_forms = build_surface_form_dict(meta)
matcher       = TextMatcher(surface_forms, meta_df=meta)
print("Text matcher ready")

In [ ]:
# ── Model config ─────────────────────────────────────────────────────────────
# We run two models; cache features for both.
# ViT-L/14 openai  (~428M params, best quality/speed tradeoff)
# SigLIP SO400M    (~400M params, strong alternative)
MODELS = [
    {"name": "ViT-L-14",             "pretrained": "openai",  "tag": "vitl14_openai",  "size": 224},
    {"name": "ViT-SO400M-14-SigLIP", "pretrained": "webli",   "tag": "siglip_so400m", "size": 224},
]
BATCH = 128

In [ ]:
# ── Extract image features for each model ────────────────────────────────────
from torch.utils.data import DataLoader
from src.data import GalleryDataset
from src.features import extract_image_features, save_features, load_features

for m in MODELS:
    feat_dir = OUT_DIR / "features" / m["tag"]
    feat_dir.mkdir(parents=True, exist_ok=True)

    # Skip if all three splits are cached
    if all((feat_dir / f"{s}_image.npy").exists() for s in ["train","val","test"]):
        print(f"  {m['tag']}: all splits cached, skipping")
        continue

    print(f"\n=== Loading {m['tag']} ===")
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms(
        m["name"], pretrained=m["pretrained"])
    model = model.to(DEVICE).eval()
    print(f"  Params: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

    splits = {
        "train": (train_df,    DATA_DIR / "train_gallery"),
        "val":   (val_gallery, DATA_DIR / "validation_gallery"),
        "test":  (test_gallery,DATA_DIR / "test_gallery"),
    }
    for split, (df, img_dir) in splits.items():
        cache = feat_dir / f"{split}_image.npy"
        if cache.exists():
            print(f"  {split}: cache hit"); continue
        ds     = GalleryDataset(df, img_dir, transform=preprocess)
        loader = DataLoader(ds, batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
        feats, ids = extract_image_features(model, loader, device=DEVICE, use_tta=True)
        save_features(feats, ids, feat_dir, split)

    del model; torch.cuda.empty_cache()

print("\nImage feature extraction complete")

In [ ]:
# ── Build category text features (prompt ensembling) ─────────────────────────
import ast
from src.features import extract_text_features, aggregate_text_features, save_text_features, load_text_features

cat_names = meta["category_name"].tolist()

# alternative names per category index
alt_names_dict = {}
for idx, row in meta.reset_index(drop=True).iterrows():
    raw = row.get("alternative_names", "")
    if isinstance(raw, str) and raw.strip():
        try:    alt_names_dict[idx] = ast.literal_eval(raw)
        except: alt_names_dict[idx] = [s.strip() for s in raw.split(",") if s.strip()]

desc_dict = {idx: str(row["description"])
             for idx, row in meta.reset_index(drop=True).iterrows()}

from src.features import build_prompt_texts
texts, cat_indices = build_prompt_texts(
    cat_names,
    alternative_names=alt_names_dict,
    descriptions=desc_dict,
)
print(f"Total prompts: {len(texts)} for {K} categories")

In [ ]:
# ── Encode text features for each model ──────────────────────────────────────
import open_clip, torch

for m in MODELS:
    feat_dir   = OUT_DIR / "features" / m["tag"]
    text_cache = feat_dir / "text.npy"
    if text_cache.exists():
        print(f"  {m['tag']}: text cache hit, skipping")
        continue

    print(f"\n=== Text features: {m['tag']} ===")
    model, _, _ = open_clip.create_model_and_transforms(
        m["name"], pretrained=m["pretrained"])
    model = model.to(DEVICE).eval()
    tokenize = open_clip.get_tokenizer(m["name"])

    def encode_texts(batch):
        tokens = tokenize(batch).to(DEVICE)
        with torch.no_grad():
            emb = model.encode_text(tokens)
        return emb.float().cpu().numpy()

    raw_feats = extract_text_features(encode_texts, texts, batch_size=256)
    cat_proto = aggregate_text_features(raw_feats, cat_indices, K)
    save_text_features(cat_proto, list(range(K)), feat_dir, tag="text")
    print(f"  category prototypes: {cat_proto.shape}")

    del model; torch.cuda.empty_cache()

print("\nText feature extraction complete")

In [ ]:
# ── Zero-shot scoring & evaluation ────────────────────────────────────────────
from src.metrics import evaluate, print_metrics
from src.ensemble import temperature_scale

def score_to_preds(S, query_df, matcher, cat_id2idx, gallery_ids, top=20):
    pred_dict = {}
    for _, row in query_df.iterrows():
        cat_id, _, _ = matcher.predict(str(row["description"]))
        cat_idx = cat_id2idx.get(cat_id, 0)
        order   = np.argsort(-S[:, cat_idx], kind="stable")
        pred_dict[str(row["query_id"])] = [gallery_ids[j] for j in order[:top]]
    return pred_dict

results_table = []

for m in MODELS:
    feat_dir = OUT_DIR / "features" / m["tag"]
    X_val, val_ids = load_features(feat_dir, "val", gallery_df=val_gallery)
    T_val, _       = load_text_features(feat_dir, tag="text")

    # Raw cosine
    S_raw  = X_val @ T_val.T
    # Softmax over categories (recommended — penalises look-alike dishes)
    S_soft = temperature_scale(S_raw, T=1.0, to_prob=True)

    preds   = score_to_preds(S_soft, val_queries, matcher, cat_id2idx, val_ids)
    metrics = evaluate(preds, val_gt)
    results_table.append({"model": m["tag"], **metrics})
    print(f"\n{m['tag']}:")
    print_metrics(metrics)

import pandas as pd
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(pd.DataFrame(results_table).to_string(index=False))

In [ ]:
# ── Best model: generate & save submission ────────────────────────────────────
# Pick the model with the best val R-precision
best = max(results_table, key=lambda x: x["r_precision"])
BEST_TAG = best["model"]
print(f"Best model: {BEST_TAG}  R-prec={best['r_precision']:.4f}")

from src.submit import build_submission, validate_submission
import datetime

feat_dir = OUT_DIR / "features" / BEST_TAG
X_test, test_ids = load_features(feat_dir, "test", gallery_df=test_gallery)
T_test, _        = load_text_features(feat_dir, tag="text")
S_test = temperature_scale(X_test @ T_test.T, T=1.0, to_prob=True)

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat_id2idx.get(cat_id, 0))

sub_df   = build_submission(S_test, test_ids, test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

(OUT_DIR / "submissions").mkdir(exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
sub_path = OUT_DIR / "submissions" / f"{ts}_zeroshot_{BEST_TAG}.csv"
sub_df.to_csv(sub_path, index=False)
sub_df.to_csv(OUT_DIR / "submission.csv", index=False)
print(f"Saved: {sub_path}")

# Official validator
import subprocess, sys
r = subprocess.run(
    [sys.executable, str(DATA_DIR/"validate_submission.py"),
     str(OUT_DIR/"submission.csv"), "--data-dir", str(DATA_DIR)],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr if r.stderr else "")
print("Validator exit code:", r.returncode)

In [ ]:
# ── Experiment log rows ───────────────────────────────────────────────────────
import datetime
today = datetime.date.today()
print("Add these rows to reports/experiment_log.md:\n")
for r in results_table:
    print(f"| E00X | {today} | 3 | {r['model']} zero-shot | prompt_ensemble, softmax, TTA=hflip, T=1.0, seed=42 | 42 | "
          f"{r['r_precision']:.4f} | {r['map@20']:.4f} | {r['p@5']:.4f} | "
          f"{r['hit@1']:.4f} | {r['hit@5']:.4f} | {r['mrr@20']:.4f} | zero-shot |")